In [ ]:
%pip install SimpleITK pandas numpy matplotlib scikit-image opencv-python
from src.utils.data_loading import loading_data
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from dotenv import load_dotenv
load_dotenv()

from matplotlib.colors import ListedColormap

from torch.utils.data import DataLoader
from src.utils.data_management import CustomDataset, Trainer
from sklearn.model_selection import train_test_split
import torch
from src.model.unet import UNet

In [ ]:
path = MC_PATH
data = loading_data(path=path)
print(data.shape)

In [ ]:
# Vérifie la forme de la première image HS
print("Forme de HS :", data['HS'].iloc[0].shape)
print("Forme de LS :", data['LS'].iloc[0].shape)
print(data['HS'].iloc[0])
print(data['LS'].iloc[0])

In [ ]:
top= mpl.colormaps.get_cmap('viridis').resampled(64)
bottom= mpl.colormaps.get_cmap('plasma').resampled(960)
newcolors = np.vstack((top(np.linspace(0, 1, 64)),
                       bottom(np.linspace(1, 0, 960))))
newcmp = ListedColormap(newcolors, name='MonteCarlo')

HS = data['HS'].iloc[0]

print(np.max(HS))
LS = data['LS'].iloc[0]

CT = data['CT'].iloc[0]



plt.figure()
plt.imshow(HS, interpolation=None, cmap=newcmp)
plt.xlabel('High sampling',fontweight='bold')
plt.colorbar(aspect=50)
plt.xticks([], [])
plt.yticks([], [])


plt.figure()
plt.imshow(LS, interpolation=None, cmap=newcmp)
plt.xlabel('Low sampling',fontweight='bold')
plt.colorbar(aspect=50)
plt.xticks([], [])
plt.yticks([], [])


plt.figure()
plt.imshow(CT, interpolation=None, cmap=newcmp)
plt.xlabel('CT',fontweight='bold')
plt.colorbar(aspect=50)
plt.xticks([], [])
plt.yticks([], [])

plt.show()


In [ ]:
X = np.array(data[["LS", "CT"]])
y = np.array(data[["HS"]])

print(X.shape)
print(y.shape)



X_rest, X_test, y_rest, y_test = train_test_split(
    X, y, train_size=0.8, random_state=42
)


X_train, X_val, y_train, y_val = train_test_split(
    X_rest, y_rest, train_size=0.8, random_state=43
)


print("X_train shape :", X_train.shape)
print("X_test shape :", X_test.shape)
print("X_val shape :", X_val.shape)

import numpy as np

# Vérifier les dimensions de chaque élément de X
for i, img in enumerate(X):
    print(f"Index {i}: forme {np.shape(img)}")

In [ ]:
train_dataloader = DataLoader(
    CustomDataset(X_train, y_train),
    batch_size=64,
    shuffle=True
)

validation_dataloader = DataLoader(
    CustomDataset(X_val, y_val),
    batch_size=64,
    shuffle=False
)

test_dataloader = DataLoader(
    CustomDataset(X_test, y_test),
    batch_size=64,
    shuffle=False
)


In [ ]:
import torch.nn as nn
import torch.optim as optim


model = UNet(n_channels=2, n_classes=1)


optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.MSELoss()


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
trainer = Trainer(
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    max_epochs=30,
    device=device
)


trainer.fit(train_dataloader, validation_dataloader)

In [ ]:
import torch
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap


top = mpl.colormaps.get_cmap('viridis').resampled(64)
bottom = mpl.colormaps.get_cmap('plasma').resampled(960)
newcolors = np.vstack((top(np.linspace(0, 1, 64)),
                       bottom(np.linspace(1, 0, 960))))
newcmp = ListedColormap(newcolors, name='MonteCarlo')

model.eval()
with torch.no_grad():
    for X_batch, y_batch in test_dataloader:
        X_batch = X_batch.to(device)
        y_pred = model(X_batch)
        break  # premier batch


idx = 38
ct_img = X_batch[idx, 0].cpu().numpy()
ls_img = X_batch[idx, 1].cpu().numpy()
ground_truth = y_batch[idx, 0].cpu().numpy()
prediction = y_pred[idx, 0].cpu().numpy()

# même echelle pour les deux images pour pouvoir comparer
vmin = min(ct_img.min(), ls_img.min(), ground_truth.min(), prediction.min())
vmax = max(ct_img.max(), ls_img.max(), ground_truth.max(), prediction.max())

# affichage
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

im0 = axes[0].imshow(ct_img, interpolation=None, cmap=newcmp, vmin=vmin, vmax=vmax)
axes[0].set_xlabel('Entrée CT', fontweight='bold', fontsize=12)
fig.colorbar(im0, ax=axes[0], aspect=50, fraction=0.046, pad=0.04)
axes[0].set_xticks([])
axes[0].set_yticks([])

im1 = axes[1].imshow(ls_img, interpolation=None, cmap=newcmp, vmin=vmin, vmax=vmax)
axes[1].set_xlabel('Entrée Low sampling (LS)', fontweight='bold', fontsize=12)
fig.colorbar(im1, ax=axes[1], aspect=50, fraction=0.046, pad=0.04)
axes[1].set_xticks([])
axes[1].set_yticks([])

im2 = axes[2].imshow(ground_truth, interpolation=None, cmap=newcmp, vmin=vmin, vmax=vmax)
axes[2].set_xlabel('Cible High sampling (HS)', fontweight='bold', fontsize=12)
fig.colorbar(im2, ax=axes[2], aspect=50, fraction=0.046, pad=0.04)
axes[2].set_xticks([])
axes[2].set_yticks([])

im3 = axes[3].imshow(prediction, interpolation=None, cmap=newcmp, vmin=vmin, vmax=vmax)
axes[3].set_xlabel('Prédiction U-Net', fontweight='bold', fontsize=12)
fig.colorbar(im3, ax=axes[3], aspect=50, fraction=0.046, pad=0.04)
axes[3].set_xticks([])
axes[3].set_yticks([])

plt.tight_layout()
plt.show()

In [ ]:
model.eval()

test_loss = 0.0

with torch.no_grad():
    for X_batch, y_batch in test_dataloader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch)
        test_loss += loss.item() * X_batch.size(0)

avg_test_loss = test_loss / len(test_dataloader.dataset)

print(f"MSE Loss moyenne sur le jeu de test : {avg_test_loss:.6f}")